# Hands-on Lab #4_0 – Support Vector Machines (SVM)

**Dataset:** [Iris – Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)  


## 1. Cargar y explorar el dataset

Iris contiene 150 flores y cuatro características. Para este ejercicio utilizaremos únicamente **Setosa y Versicolor** (100 muestras), con dos características: **longitud y ancho del pétalo**. Así podremos visualizar la clasificación en dos dimensiones.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

iris = load_iris()
X = pd.DataFrame(iris.data[:, 2:4], columns=['petal_length', 'petal_width'])
y = pd.Series(iris.target, name='species')

mask = y.isin([0, 1])  # 0 = Setosa, 1 = Versicolor
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print('Dimensiones:', X.shape)
print('Clases:', iris.target_names[:2])
print('Distribución:', y.value_counts().sort_index().to_dict())
display(X.head())

**Resultado esperado:** `(100, 2)`; 50 flores por clase.  


## 2. Visualizar las dos clases

In [ ]:
for etiqueta, nombre, color in [(0, 'Setosa', 'tab:blue'), (1, 'Versicolor', 'tab:green')]:
    grupo = X[y == etiqueta]
    plt.scatter(grupo['petal_length'], grupo['petal_width'],
                label=nombre, color=color, edgecolor='k', alpha=0.8)
plt.xlabel('Longitud del pétalo (cm)')
plt.ylabel('Ancho del pétalo (cm)')
plt.title('Iris: clasificación binaria')
plt.legend()
plt.show()

## 3. Dividir los datos en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

## 4. Entrenar un SVM lineal

Usaremos `StandardScaler` dentro de un `Pipeline`. El escalador aprende sus parámetros únicamente con los datos de entrenamiento para evitar fuga de información (*data leakage*).

In [ ]:
modelo = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1))
])
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=iris.target_names[:2]))
print('Accuracy:', accuracy_score(y_test, y_pred))

## 5. Matriz de confusión

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=iris.target_names[:2], cmap='Blues'
)
plt.title('SVM lineal – Matriz de confusión')
plt.show()

**Actividad:** Identifica las predicciones correctas y las flores clasificadas incorrectamente, si las hubiera.

## 6. Frontera de decisión, márgenes y vectores de soporte

El SVM se entrenó con características estandarizadas. Para graficar en **centímetros**, transformamos la cuadrícula con el mismo escalador. Las líneas corresponden a los valores de la función de decisión `−1`, `0` y `+1`.

In [ ]:
escalador = modelo.named_steps['scaler']
svc = modelo.named_steps['svm']

x_min, x_max = X['petal_length'].min() - 0.3, X['petal_length'].max() + 0.3
y_min, y_max = X['petal_width'].min() - 0.3, X['petal_width'].max() + 0.3
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350),
                     np.linspace(y_min, y_max, 350))
cuadricula = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X.columns)

prediccion = modelo.predict(cuadricula).reshape(xx.shape)
decision = svc.decision_function(escalador.transform(cuadricula)).reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, prediccion, alpha=0.15, cmap='coolwarm')
plt.contour(xx, yy, decision, levels=[-1, 0, 1],
            colors='black', linestyles=['--', '-', '--'])
plt.scatter(X_train['petal_length'], X_train['petal_width'],
            c=y_train, cmap='coolwarm', edgecolors='k', label='Muestras de entrenamiento')

# Los vectores de soporte están en coordenadas estandarizadas: volver a cm.
soportes_originales = escalador.inverse_transform(svc.support_vectors_)
plt.scatter(soportes_originales[:, 0], soportes_originales[:, 1],
            s=170, facecolors='none', edgecolors='red', linewidths=2,
            label='Vectores de soporte')
plt.xlabel('Longitud del pétalo (cm)')
plt.ylabel('Ancho del pétalo (cm)')
plt.title('SVM lineal: frontera de decisión y márgenes')
plt.legend()
plt.show()

print('Vectores de soporte por clase:', svc.n_support_)
print('Total de vectores de soporte:', len(svc.support_vectors_))